# Home Exercise: Predicting Diabetes

**The task.** The **Pima Indians Diabetes** dataset describes 768 women of Pima heritage, aged 21 and
over. For each woman we have 8 medical measurements, and whether she was diagnosed with **diabetes**
within five years (`outcome = 1`). Your job is to build, evaluate and interpret a logistic regression
model for this prediction, **on your own**, using everything from the unit.

**How this exercise works**

- This is **independent work**. Each task says *what* to do, not *how*. Reuse code from the notebooks
  freely.
- There are a few automatic **check cells** (✅ / ❌) for the parts where everyone should get the same
  numbers. Most parts have no single right answer: you'll be graded on **correct method** and
  **honest, well-reasoned explanations**, not on getting the highest score.
- Where a task asks a question **in words**, answer in the markdown cell under it: 2 to 5 sentences.
- Expected time: 3 to 4 hours. Don't leave it all for the last evening.

**Grading:** see the points next to each task (100 in total).

**The features**

| column | meaning |
|---|---|
| `pregnancies` | number of pregnancies |
| `glucose` | blood glucose concentration two hours into a glucose tolerance test |
| `blood_pressure` | diastolic blood pressure (mm Hg) |
| `skin_thickness` | triceps skin-fold thickness (mm) |
| `insulin` | 2-hour serum insulin (µU/ml) |
| `bmi` | body mass index (weight in kg / (height in m)²) |
| `dpf` | "diabetes pedigree function": a score of diabetes in the family history |
| `age` | age in years |
| `outcome` | 1 = diagnosed with diabetes within 5 years, 0 = not |

*Source:* originally from the US National Institute of Diabetes and Digestive and Kidney Diseases, via
the UCI Machine Learning Repository.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# Interactive sliders (optional): if ipywidgets is missing, we show a few fixed pictures instead.
try:
    from ipywidgets import interact, FloatSlider, FloatLogSlider
    HAVE_WIDGETS = True
except ImportError:
    HAVE_WIDGETS = False
    print("ipywidgets is not installed: the interactive cells will show fixed pictures instead.")

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report, roc_auc_score,
                             precision_score, recall_score, f1_score)

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

In [ ]:
# --- exercise checker: just run this cell -----------------------------------------------
import base64 as _b64, json as _json
_ANS = _json.loads(_b64.b64decode("eyIxYSI6IFs3NjgsIDAuMzQ4OTU4MzMzMzMzMzMzM10sICIxYiI6IFs1LCAzNSwgMjI3LCAzNzQsIDExXSwgIjJhIjogWzU3NiwgMTkyXSwgIjJiIjogWzExNy4wLCA3Mi4wLCAzMC4wLCAxMjUuMCwgMzIuMF0sICIzYiI6IDAuNjUxMDQxNjY2NjY2NjY2N30="))

def _has_str(v):
    return isinstance(v, str) or (isinstance(v, (list, tuple)) and any(_has_str(x) for x in v))

def _norm(v):
    return [_norm(x) for x in v] if isinstance(v, (list, tuple, np.ndarray)) else str(v).strip().lower()

def _close(a, b, tol):
    if _has_str(b):
        return _norm(a) == _norm(b)
    try:
        return np.allclose(np.asarray(a, dtype=float), np.asarray(b, dtype=float), atol=tol, rtol=0)
    except Exception:
        return False

def check(key, value, tol=1e-3):
    # Compare your answer for exercise `key` with the expected one.
    if value is None or value is Ellipsis or (isinstance(value, (list, tuple)) and any(v is None for v in value)):
        print(f"⏳ {key}: not answered yet")
    elif _close(value, _ANS[key], tol):
        print(f"✅ {key}: correct!")
    else:
        print(f"❌ {key}: not yet. You got {value}; check your work and try again")

def check_against(key, value, reference, tol=1e-3):
    # Compare your answer with a reference value computed in the same cell.
    if value is None or (isinstance(value, (list, tuple)) and any(v is None for v in value)):
        print(f"⏳ {key}: not answered yet")
    elif _close(value, reference, tol):
        print(f"✅ {key}: correct!")
    else:
        print(f"❌ {key}: not yet. You got {value}; check your work and try again")

def check_fn(key, fn, tol=1e-3):
    # Call your function on some test inputs and compare with the expected outputs.
    spec = _ANS[key]
    try:
        outs = [fn(*[np.array(a, dtype=float) for a in args]) for args in spec["args"]]
    except Exception as e:
        print(f"❌ {key}: your function raised an error: {type(e).__name__}: {e}")
        return
    if any(o is None for o in outs):
        print(f"⏳ {key}: not answered yet (your function returns None)")
    elif all(_close(o, ex, tol) for o, ex in zip(outs, spec["out"])):
        print(f"✅ {key}: correct!")
    else:
        print(f"❌ {key}: not yet. Your function gives {outs}; check your work and try again")

def check_plausible(name, value, low, high):
    # for results that depend on your choices: is the number in a reasonable range?
    if value is None:
        print(f"⏳ {name}: not computed yet")
    elif low <= value <= high:
        print(f"✅ {name} = {value:.3f} is in a plausible range")
    else:
        print(f"❓ {name} = {value:.3f} is outside the range we'd expect ({low} to {high}). "
              "Recheck your steps; if you're confident, explain why in words.")

In [ ]:
diabetes = pd.read_csv("../data/pima_diabetes.csv")
diabetes.head()

## Task 1: Explore the data (10 points)

**1a.** How many women are in the dataset, and what fraction of them have `outcome = 1`? Store them in
`t1a = [n, fraction]`. Is the data balanced?

In [ ]:
# YOUR CODE HERE
t1a = None
print(t1a)
check("1a", t1a)

**1b.** Look at `diabetes.describe()`. Some columns contain values that are **medically
impossible**. (Can a living person have a BMI of 0? A blood pressure of 0?)

1. Find these columns.
2. For each of the five columns `glucose`, `blood_pressure`, `skin_thickness`, `insulin` and `bmi`,
   count how many rows have the value 0. Store the five counts, in that order, in `t1b`.
3. **In words:** what do you think these zeros really mean? Why would they be a problem for a
   logistic regression model if we left them as they are?

In [ ]:
# YOUR CODE HERE
t1b = None
print(t1b)
check("1b", t1b)

**Your answer:** *(write here)*

## Task 2: Split and clean (10 points)

**2a.** Separate the features `X` (all columns except `outcome`) from the labels `y`. Split them into a
training part and a test part with **exactly these settings**, so that everyone's results are
comparable: `test_size=0.25, stratify=y, random_state=0`. Name the results
`X_train, X_test, y_train, y_test`, and store their sizes in `t2a = [n_train, n_test]`.

From now on, **don't look at the test set** until a task explicitly says so.

In [ ]:
# YOUR CODE HERE
t2a = None
check("2a", t2a)

**2b.** Replace the impossible zeros with a reasonable value: the **median** of that column,
computed from the **training data only**, and ignoring the zeros themselves (they aren't real values).

1. Compute the five medians, in the order of Task 1b, and store them in `t2b`.
2. Write a function `impute(df, medians)` that returns a **copy** of `df` with the zeros in those five
   columns replaced by the medians. Apply it to both `X_train` and `X_test`, and call the results
   `X_train_c` and `X_test_c`.
3. **In words:** why must the medians come from the training data only? (Hint: notebook 02, the
   scaler.)

*Tip:* `df[col].replace(0, np.nan)` turns the zeros into "missing" values (`NaN`), which `.median()`
ignores, and `.fillna(value)` fills them in.

In [ ]:
# YOUR CODE HERE
t2b = None
print(t2b)
check("2b", t2b)

**Your answer:** *(write here)*

## Task 3: A baseline model (15 points)

**3a.** Build a model that scales the features and fits a `LogisticRegression` with default settings
(a pipeline is the easiest way, as in notebook 04). Train it on the **cleaned** training data. Call it
`model`.

**3b.** A "lazy" classifier predicts **no diabetes for everyone**. What is its accuracy on the test
set? Store it in `t3b`.

**3c.** Now evaluate `model` on the (cleaned) **test** set: accuracy, confusion matrix, precision,
recall, F1 and AUC, and the `classification_report`. Store the test AUC in `t3c_auc`.

**3d. In words:** compare your model with the lazy classifier. Is the model's accuracy impressive? Which
measure tells you the most about how useful it is for **finding** women who will develop diabetes?

In [ ]:
# YOUR CODE HERE
t3b = None
t3c_auc = None
check("3b", t3b)
check_plausible("test AUC", t3c_auc, 0.80, 0.90)

**Your answer:** *(write here)*

**3e. Did the cleaning help?** Compare the model trained on the **raw** data (zeros left in) with
the model trained on the **cleaned** data. Don't use the test set for this. Instead, repeat the
following 20 times, with `random_state` = 0 to 19:

- split the **training** data into a training part and a validation part (30% validation, stratified);
- compute the medians from the training part only, and impute both parts;
- fit both models (raw and cleaned) on the training part, and compute their validation AUCs.

Report the average validation AUC of each, and in how many of the 20 splits the cleaned model won.

**In words:** does cleaning help? What would you have concluded if you had compared the two models on
the **test set alone**? (Try it and see!)

In [ ]:
# YOUR CODE HERE

**Your answer:** *(write here)*

## Task 4: Interpret the model (10 points)

Print the weights of `model` next to the feature names. (With a pipeline, the logistic regression is
the last step: `model[-1].coef_[0]`.)

**In words:**

1. Which measurements matter most for the prediction? Does that match what you know, or can look up,
   about diabetes?
2. Which feature has almost no effect? Does that mean it's medically irrelevant? (Careful: think about
   correlated features.)
3. For the most important feature, by what factor do the **odds** of diabetes multiply when that
   feature increases by one standard deviation? (01c, Part E.)

In [ ]:
# YOUR CODE HERE

**Your answer:** *(write here)*

## Task 5: A screening threshold (15 points)

Suppose the model is used for **screening**: women the model flags will be sent for a full diagnostic
test. The clinic wants to catch **at least 80%** of the women who will develop diabetes (recall
$\ge 0.8$).

1. Choose the threshold **without the test set**: split the training data into training and validation
   parts (30% validation, stratified, `random_state=0`), clean, fit, and find the **largest** threshold
   (in steps of 0.01) whose **validation** recall is at least 0.8.
2. Why the *largest* such threshold?
3. **Now** apply that threshold to `model` on the **test** set, once. Report the test recall and
   precision.
4. **In words:** did the test recall reach 0.8? If not, why not, and what would you tell the clinic?

In [ ]:
# YOUR CODE HERE

**Your answer:** *(write here)*

## Task 6: Regularization (10 points)

Using validation splits of the **training** data (as in 03b: 10 splits, 30% validation, cleaning and
scaling inside each split), compare the average validation AUC for
$C \in \{0.001, 0.01, 0.1, 1, 10, 100\}$. Plot it.

**In words:** does `C` matter much for this dataset? Explain the result using what you know about the
number of examples and the number of features.

In [ ]:
# YOUR CODE HERE

**Your answer:** *(write here)*

## Task 7: Polynomial features (10 points)

Do degree-2 polynomial features help here? Compare a linear model with a degree-2 polynomial model
(scaling, then `PolynomialFeatures(degree=2, include_bias=False)`, then `LogisticRegression`), using the
same kind of validation splits as in Task 6. Try a couple of values of `C` for the polynomial model.

**In words:** give an **honest** verdict. If polynomial features don't help, say so, and suggest why,
based on notebook 04.

In [ ]:
# YOUR CODE HERE

**Your answer:** *(write here)*

## Task 8: Your own implementation (10 points)

Use **your own** gradient-descent implementation from notebooks 03a/03b (paste your `fit_reg`) to
train on the **cleaned, scaled** training data, with the $\lambda$ that corresponds to `C=1`. Compare
your weights with `model[-1].coef_[0]`. The largest difference should be below 0.01. (If it isn't:
more steps? a different learning rate? Did you scale the same way?)

In [ ]:
# YOUR CODE HERE

## Task 9: Conclusion (10 points)

Write a short summary (5 to 8 sentences) for a **doctor** who knows medicine but not machine learning:

- How well does the model work? (Use a number the doctor can understand.)
- What does it rely on most?
- How should it be used, and what should it **not** be used for?
- What are its limitations? (Think about the data: who is in it? What was missing?)

**Your answer:** *(write here)*

---
*Before you submit:* restart the kernel and run all the cells (Kernel → Restart Kernel and Run All
Cells). Make sure every cell runs without errors, and that every "in words" answer is filled in.